# Paper 16: Honest regime throttle

Compare confirmation days, hysteresis bands, partial throttles, and a volatility target against always invested. The v2 baseline switch is still same-close, next-open, no confirmation.

Scored against **v2 baseline**. Kill rules are frozen in `Research/book-construction-status.md`:

- Train 2022 calendar more than 5pp worse than SPUS
- Train max drawdown worse than SPUS
- Train Calmar below both v2 baseline and SPUS
- Test 2023 calendar more than 15pp worse than v2 baseline

**Variants.** always-on, confirm-2, confirm-5, band-2pct, partial-50, vol-12.


In [ ]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "monterey" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError(f"Monterey repo root not found from {start}")

_ROOT = _repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
try:
    import monterey  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{_ROOT}[research]"])

from monterey.research import boot, compare
from monterey.sprints import run_sprint, score_against_baseline

rd = boot("paper-16-honest-throttle", start="2019-12-01", end="2026-09-30")
print("snapshot", rd.manifest.get("hash"))


In [ ]:
ledgers, table = run_sprint("16", rd)
table


In [ ]:
baseline, *rest = ledgers
for book in rest:
    verdict = score_against_baseline(book, baseline, rd)
    print(book.spec.id, "KILLED" if verdict["killed"] else "keep", "; ".join(verdict["reasons"]) or "pass")
